# Byte-Pair Encoding (BPE)

## Задание 1
У вас есть готовая функция `get_pairs()` для получения всех последовательных пар. Напишите код функции `best_pair()`, которая получает список слов, где каждое слово —  список токенов. Функция `best_pair()` с помощью функции `get_pairs()` должна возвращать самую частотную пару и её частотность.
Далее реализуйте функцию `merge_pair()`, которая объединяет все вхождения пары pair в новый токен. Например, для списка символов `["a", "b", "c", "a"]` функция get_pairs вернет `[("a","b"), ("b","c"), ("c","a")]`, а `merge_pair(symbols, ("b","c"))` должна вернуть `["a","bc","a"]`. Используйте циклы по индексам, как в примере выше.

In [5]:
from collections import Counter

def get_pairs(symbols):
    """
    Возвращает список кортежей соседних пар в списке символов.
    Пример: ["h","e","l","l","o"] -> 
    [("h","e"),("e","l"),("l","l"),("l","o")].
    """
    pairs = []
    for i in range(len(symbols)-1):
        pairs.append((symbols[i], symbols[i+1]))
    return pairs

def best_pair(words):
    """
    Находит пару (bigram), которую нужно слить на следующем шаге BPE:
    выбирает самую частотную соседнюю пару по всем словам.

    words: список слов, где каждое слово — список символов/токенов.
           например: [["a","b","a","b","c"], ["a","b","c"]]
    Возвращает: (пара, частота) или (None, 0), если пар нет.
    """
    pairs = []
    for word in words:
        pairs.extend(get_pairs(word))
    pair_counts = Counter(pairs)
    if len(pair_counts) == 0:
        return None, 0
    return pair_counts.most_common(1)[0]
    

def merge_pair(symbols, pair):
    """
    Объединяет все вхождения заданной пары в новый токен.
    Пример: ["h","e","l","l","o"], pair=("l","l") 
    -> ["h","e","ll","o"].
    """
    new_symbols = []
    i = 0
    while i < len(symbols):
        if tuple(symbols[i:i+2]) == pair:
            new_symbols.append(pair[0] + pair[1])
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
    return new_symbols


words = [
    ["a", "b", "c", "a"],
    ["a", "b", "c"],
    ["b", "c", "a"]
]

pair, freq = best_pair(words)
print(pair, freq)

words_merged = [merge_pair(w, pair) for w in words]
print(words_merged)

('b', 'c') 3
[['a', 'bc', 'a'], ['a', 'bc'], ['bc', 'a']]


# WordPiece

## Задание 2
В прекоде заданы основные функции:
- `get_pairs_wp()` — возвращает список пар токенов;
- `merge_pair_wp()` — объединяет вхождения пары в токен;

… и вспомогательные функции:
- `wp_strip()` — убирает префиксы, если они есть;
- `wp_prefix()` — выделяет префикс токена;
- `init_wordpiece_tokens()` — делит слово на изначальные токены — символы с префиксом ##, если они стоят не в начале слова.

Чтобы всё заработало, вам нужно реализовать функцию best_pair_wp(), которая вернёт пару с наибольшим значением $score$ (о нём вы узнали чуть раньше).

In [21]:
from collections import Counter

def wp_strip(tok: str) -> str:
    """Убирает префикс ## если он есть."""
    return tok[2:] if tok.startswith("##") else tok

def wp_prefix(tok: str) -> str:
    """Если токен продолжения слова — возвращаем '##', иначе ''."""
    return "##" if tok.startswith("##") else ""

def get_pairs_wp(symbols):
    """
    Соседние пары как в BPE.
    Пример: ["h","##e","##l"] -> [("h","##e"),("##e","##l")]
    """
    pairs = []
    for i in range(len(symbols) - 1):
        pairs.append((symbols[i], symbols[i+1]))
    return pairs

def merge_pair_wp(symbols, pair):
    """
    Объединяет все вхождения пары в новый WordPiece-токен,
    корректно обрабатывая '##'.

    Правило:
    - если первый токен пары начинается с '##', то и результат начинается с '##'
    - при склейке удаляем '##' у обоих частей и склеиваем "чистые" строки
    """
    a, b = pair
    new_symbols = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == pair:
            pref = wp_prefix(symbols[i])
            merged = pref + wp_strip(symbols[i]) + wp_strip(symbols[i+1])
            new_symbols.append(merged)
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
    return new_symbols

def best_pair_wp(words):
    """
    Выбирает пару для слияния по скору WordPiece:
        score(a,b) = count(a b) / (count(a) * count(b))

    words: список слов, где слово — список wp-токенов (с ## для продолжений)
    Возвращает: (pair, score, pair_freq) или (None, 0.0, 0)
    """
    symbols, pairs = [], []
    for word in words:
        symbols.extend(word)
        pairs.extend(get_pairs_wp(word))
    symbols_counts = Counter(symbols)
    pair_counts = Counter(pairs)
    if len(pair_counts) == 0:
        return None, 0., 0
    scores = {(a, b): pair_counts[(a, b)] / symbols_counts[a] / symbols_counts[b] for a, b in pair_counts}
    best_pair = max(scores, key=scores.get)
    return best_pair, scores[best_pair], pair_counts[best_pair]    


def init_wordpiece_tokens(word: str):
    if not word:
        return []
    return [word[0]] + [f"##{ch}" for ch in word[1:]]

words = [
    init_wordpiece_tokens("abab"),
    init_wordpiece_tokens("abac"),
    init_wordpiece_tokens("ab"),
]

print(words)

pair, score, freq = best_pair_wp(words)
print("best:", pair, "score:", score, "freq:", freq)

words2 = [merge_pair_wp(w, pair) for w in words]
print(words2)

[['a', '##b', '##a', '##b'], ['a', '##b', '##a', '##c'], ['a', '##b']]
best: ('##a', '##c') score: 0.5 freq: 1
[['a', '##b', '##a', '##b'], ['a', '##b', '##ac'], ['a', '##b']]
